In [ ]:
!pip install -q openai google-genai
import json, time
from collections import Counter
from openai import OpenAI
from google import genai
from google.genai import types
from google.colab import userdata
from tqdm.auto import tqdm

oai = OpenAI(api_key=userdata.get('WORKIT_API'))
gem = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
O3_MODEL = 'o3'
GEMINI_MODEL = 'gemini-3.1-flash-lite'

In [ ]:
# ============================================================
# [유틸] 공통 헬퍼 함수들
# ============================================================

def chunks(lst, n):
    """리스트를 n개씩 잘라 배치로 넘겨주는 제너레이터. (예: 50건 → 5건씩 10배치)"""
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def get_nested(d, path):
    """'output.label' 같은 중첩 경로로 값 꺼내기. CON은 'label'(최상위), PEP/RPT는 'output.label'(중첩)이라 필요."""
    cur = d
    for k in path.split('.'):
        cur = cur.get(k) if isinstance(cur, dict) else None
    return cur

def parse_arr(txt):
    """LLM 응답 텍스트(JSON)를 배열로 변환.
    - 최상위가 배열이면 그대로
    - {"results":[...]}처럼 객체로 감싸여 오면 그 안의 배열을 꺼냄
    - 그마저 없으면 객체 하나를 배열에 담아 반환"""
    if not txt: return []
    p = json.loads(txt)
    if isinstance(p, list): return p
    for v in p.values():
        if isinstance(v, list): return v
    return [p]


# ============================================================
# [지표] IAA 측정용 (순수 파이썬, sklearn 불필요)
# ============================================================

def cohen_kappa(g, pr, labels):
    """Cohen's kappa 계산. g=내 골든 라벨, pr=AI 판정.
    우연히 맞을 확률(pe)을 제거한 일치도. 1에 가까울수록 내 기준과 AI가 잘 맞음.
    반환: (kappa, 단순일치율 po, 비교한 건수 n)"""
    ids = [i for i in g if i in pr]; n = len(ids)
    if n == 0: return None, 0.0, 0
    po = sum(1 for i in ids if g[i]==pr[i]) / n          # 실제 일치 비율
    pe = 0.0
    for L in labels:
        pe += (sum(1 for i in ids if g[i]==L)/n) * (sum(1 for i in ids if pr[i]==L)/n)  # 우연 일치 기대치
    return ((po-pe)/(1-pe) if pe < 1 else 1.0), po, n

def interp(k):
    """kappa 값을 사람이 읽을 등급으로 변환 (0.8↑ 거의완벽 / 0.6↑ 상당 / 0.4↑ 보통 / 그 이하 낮음)."""
    if k is None: return 'N/A'
    return ('almost-perfect(>=0.8)' if k>0.8 else 'substantial(0.6-0.8)' if k>0.6
            else 'moderate(0.4-0.6)' if k>0.4 else 'low(<0.4)')

def confusion(g, pr, labels):
    """혼동행렬. 행=내 골든, 열=AI 판정. 어디서 엇갈리는지(예: 내가 검토인데 AI는 일치) 파악용."""
    m = {a:{b:0 for b in labels} for a in labels}
    for i in g:
        if i in pr and g[i] in labels and pr[i] in labels:
            m[g[i]][pr[i]] += 1
    return m


# ============================================================
# [IAA] 재판정 프롬프트 + 입력 구성
# ============================================================

def _instruction(labels, boundary, criteria_rules):
    """AI에게 줄 판정 지시문 생성.
    내 판정 기준(criteria_rules=RULES)을 넣어 '내 기준대로' 판정하게 하고,
    정답 label은 숨겨서(앵커링 방지) 독립 재판정시킴. 출력은 판정+이유만."""
    lab = '|'.join(labels)
    return (
        '너는 공공 SW 문서 검토자다. 각 건의 입력만 보고 아래 기준으로 판정하라.\n'
        f'- 판정값은 {labels} 중 하나다.\n'
        '- 근거는 주어진 입력 안에만 둔다. 외부 지식으로 없는 기준을 채우지 마라.\n'
        f'- {boundary}\n'
        f'{criteria_rules}\n'
        '- 입력에 정답 label은 없다. 위 기준으로 독립 판정한다.\n'
        f'출력은 JSON 배열만: [{{"id":"...","판정":"{lab}","이유":"한 문장 근거"}}]'
    )

def _payload(batch, input_fields):
    """레코드에서 AI에 줄 입력 필드만 뽑아 정리. (정답 label은 절대 안 넣음)
    CON은 [clause_text, refs], PEP/RPT는 [criteria, 발췌문들]."""
    out = []
    for it in batch:
        item = {'id': it['seed_id']}   # ★수정: it['id'](섹션코드, 중복됨) → it['seed_id'](고유값)
        for f in input_fields:
            item[f.split('.')[-1]] = get_nested(it, f) if '.' in f else it.get(f, '')
        out.append(item)
    return out


# ============================================================
# [IAA] o3 / Gemini 각각 재판정 호출
# ============================================================

def judge_o3(batch, instr, input_fields, retries=3):
    """o3(OpenAI)로 한 배치 재판정. 실패 시 최대 3회 재시도. 판정 배열 반환."""
    for a in range(retries):
        try:
            r = oai.chat.completions.create(
                model=O3_MODEL,
                messages=[{'role':'system','content':instr},
                          {'role':'user','content':json.dumps(_payload(batch,input_fields), ensure_ascii=False)}],
                response_format={'type':'json_object'}, max_completion_tokens=8000)
            txt = r.choices[0].message.content
            if not txt: time.sleep(3); continue   # 빈 응답이면 재시도
            return parse_arr(txt)
        except Exception as e:
            print('  [o3]', e); time.sleep(5)
    return []

def judge_gemini(batch, instr, input_fields, retries=3):
    """Gemini로 한 배치 재판정. o3와 동일 로직, SDK만 다름."""
    for a in range(retries):
        try:
            r = gem.models.generate_content(
                model=GEMINI_MODEL,
                contents=instr + '\n\n입력:\n' + json.dumps(_payload(batch,input_fields), ensure_ascii=False),
                config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
            if not r.text: time.sleep(3); continue
            return parse_arr(r.text)
        except Exception as e:
            print('  [gemini]', e); time.sleep(5)
    return []

def run_model(judge_fn, name, data, instr, input_fields, labels, batch):
    """한 모델(o3 or Gemini)로 전체 데이터를 배치로 돌려 판정+이유 수집.
    반환: (pred={id:판정}, reasons={id:이유})"""
    pred = {}
    reasons = {}
    for b in tqdm(list(chunks(data, batch)), desc=name, unit='batch'):
        for v in judge_fn(b, instr, input_fields):
            if 'id' in v and v.get('판정') in labels:
                pred[v['id']] = v['판정']
                reasons[v['id']] = v.get('이유', '')   # AI가 왜 그렇게 판정했는지
        time.sleep(2)
    return pred, reasons


# ============================================================
# [IAA 메인] o3 vs Gemini 비교 → 승자(검수기) 선정
# ============================================================

def iaa_compare(data, gold_field, labels, input_fields, boundary, criteria_rules, task):
    """확정 골든을 o3·Gemini 둘 다 재판정 → 각각 내 골든과 kappa 비교 → 승자 반환.
    - 각 모델의 accuracy/kappa/혼동행렬 출력
    - mismatch(내 골든≠AI) 건은 이유까지 화면에 찍음 (골든 재점검용)
    - 전체 판정+이유를 {task}_{모델}_all.json으로 저장
    - kappa 높은 모델 = 내 기준과 더 잘 맞는 검수기 → QC에 사용"""
    gold = {r['seed_id']: (get_nested(r, gold_field) if '.' in gold_field else r.get(gold_field)) for r in data}   # ★수정: r['id'] → r['seed_id']
    instr = _instruction(labels, boundary, criteria_rules)
    p_o3, rz_o3 = run_model(judge_o3, f'{task} o3', data, instr, input_fields, labels, 5)
    p_gm, rz_gm = run_model(judge_gemini, f'{task} Gemini', data, instr, input_fields, labels, 10)
    for nm, pr, rz in [('o3', p_o3, rz_o3), ('Gemini', p_gm, rz_gm)]:
        k, po, n = cohen_kappa(gold, pr, labels)
        print(f'\n== [{task}] IAA my-gold vs {nm} (n={n}) ==')
        print(f'  accuracy={po:.3f} kappa={k:.3f} -> {interp(k)}')
        m = confusion(gold, pr, labels)
        print('        ' + ''.join(f'{l:>6}' for l in labels))
        for g in labels:
            print(f'  {g:>4} ' + ''.join(f'{m[g][p]:>6}' for p in labels))
        diff = [(i, gold[i], pr[i]) for i in gold if i in pr and gold[i]!=pr[i]]
        print(f'  mismatch {len(diff)}:')
        for i, g, p in diff:
            print(f'    {i}: my={g} / {nm}={p}  ← 이유: {rz.get(i,"")}')
        json.dump({i:{'판정':pr[i],'이유':rz.get(i,'')} for i in pr},
                  open(f'{task}_{nm}_all.json','w'), ensure_ascii=False, indent=2)
    k_o3,_,_ = cohen_kappa(gold, p_o3, labels)
    k_gm,_,_ = cohen_kappa(gold, p_gm, labels)
    winner = 'o3' if (k_o3 or 0) >= (k_gm or 0) else 'Gemini'   # kappa 높은 쪽 승
    print(f'\n>>> [{task}] IAA winner: {winner}  (o3={k_o3:.3f} / Gemini={k_gm:.3f}) -> QC uses {winner}')
    return winner


# ============================================================
# [QC] 증강본 검수 → accept/review/drop 3분류
# ============================================================

def run_qc(aug, qc_prompt, winner, task, batch=5):
    """IAA 이긴 모델로 증강본을 JUDGE 프롬프트로 검수.
    각 증강 케이스를 3버킷으로 분류:
    - drop: 명백 결함 → 자동 탈락
    - review: 라벨 이견 또는 유실 → 사람이 확인
    - accept: 전부 통과 → 학습 데이터로 채택"""

    from collections import Counter, defaultdict

    # id 중복 제거 (모델에 보내기 전에 미리 유니크하게)
    id_counts = Counter(r['id'] for r in aug)
    dup_ids = {k: v for k, v in id_counts.items() if v > 1}
    if dup_ids:
        print(f'[{task}] 중복 id {len(dup_ids)}종 발견 (예: {list(dup_ids.items())[:3]}) -> #n 접미사로 유니크화')
    seen = defaultdict(int)
    fixed_aug = []
    for r in aug:
        rid = r['id']
        if id_counts[rid] > 1:
            seen[rid] += 1
            new_r = dict(r)
            new_r['id'] = f'{rid}#{seen[rid]}'
            fixed_aug.append(new_r)
        else:
            fixed_aug.append(r)
    aug = fixed_aug

    def qc_batch(b, retries=3):
        msg = json.dumps(b, ensure_ascii=False)
        need_ids = {it['id'] for it in b}
        last_partial = []
        for a in range(retries):
            try:
                if winner == 'o3':
                    r = oai.chat.completions.create(
                        model=O3_MODEL,
                        messages=[{'role':'system','content':qc_prompt},
                                  {'role':'user','content':'[입력]\n'+msg}],
                        response_format={'type':'json_object'}, max_completion_tokens=25000)
                    choice = r.choices[0]
                    txt = choice.message.content
                    if choice.finish_reason == 'length':
                        print(f'  [qc] TRUNCATED (finish_reason=length), retry {a+1}/{retries}')
                        time.sleep(3); continue
                else:
                    r = gem.models.generate_content(
                        model=GEMINI_MODEL,
                        contents=qc_prompt + '\n\n[입력]\n' + msg,
                        config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
                    txt = r.text
                if not txt: time.sleep(3); continue

                results = parse_arr(txt)
                got_ids = {v['id'] for v in results if 'id' in v}
                missing = need_ids - got_ids
                last_partial = results

                if missing and a < retries - 1:
                    print(f'  [qc] 누락 id {len(missing)}건 (attempt {a+1}/{retries}): {missing} -> 재시도')
                    time.sleep(3); continue

                return results
            except Exception as e:
                print('  [qc] ERROR', repr(e), f'(attempt {a+1}/{retries})'); time.sleep(5)

        print(f'  [qc] BATCH LOST after {retries} retries - ids: {[it.get("id") for it in b]}')
        return last_partial

    verdicts = {}
    lost_ids = []
    for b in tqdm(list(chunks(aug, batch)), desc=f'{task} QC({winner})', unit='batch'):
        results = qc_batch(b)
        got_ids = {v['id'] for v in results if 'id' in v}
        for v in results:
            if 'id' in v: verdicts[v['id']] = v
        for it in b:
            if it['id'] not in got_ids:
                lost_ids.append(it['id'])
        time.sleep(3)

    print(f'[{task}] total_input={len(aug)} judged={len(verdicts)} lost={len(lost_ids)}')
    if lost_ids:
        json.dump(lost_ids, open(f'{task}_lost_ids.json','w'), ensure_ascii=False, indent=2)

    by = {r['id']: r for r in aug}; acc, rev, drop = [], [], []
    for rid, v in verdicts.items():
        rec = by.get(rid)
        if rec is None: rev.append({'id': rid, '문제점': ['id not found']}); continue
        문제 = v.get('문제점', []); 통과 = v.get('통과', False)
        라벨OK = v.get('라벨일치', v.get('판정일치', True)) and v.get('label일치', True)
        결함 = (v.get('근거진위', True) is False) or bool(v.get('금지어', [])) or (v.get('disclaimer', True) is False) \
               or (v.get('형식정상', True) is False) or (v.get('사유진위', True) is False) or (v.get('방향성정상', True) is False)
        if 결함: drop.append({'id': rid, '사유': 문제 or ['format/basis defect']})
        elif not 라벨OK: rev.append({'id': rid, '재판정': v.get('재판정', v.get('재계산label','')), '문제점': 문제})
        elif 통과: acc.append(rec)
        else: rev.append({'id': rid, '문제점': 문제 or ['not passed']})

    judged_ids = set(verdicts.keys())
    for it in aug:
        if it['id'] not in judged_ids:
            rev.append({'id': it['id'], '문제점': ['no verdict returned (api/parse failure)']})

    total_out = len(acc) + len(rev) + len(drop)
    print(f'[{task}] accept {len(acc)} / review {len(rev)} / drop {len(drop)}  (합계={total_out}, 입력={len(aug)})')
    if total_out != len(aug):
        print(f'  경고: 합계가 입력 건수와 다릅니다.')

    json.dump(acc, open(f'{task}_aug_accept.json','w'), ensure_ascii=False, indent=2)
    json.dump(rev, open(f'{task}_aug_review.json','w'), ensure_ascii=False, indent=2)
    json.dump(drop, open(f'{task}_aug_drop.json','w'), ensure_ascii=False, indent=2)
    print(f'-> {task}_aug_accept/review/drop.json saved')

In [ ]:
from collections import Counter
c = Counter(r['id'] for r in rpt_aug)
print({k:v for k,v in c.items() if v>1})

In [ ]:
raw = json.load(open('RPT_train (11).json', encoding='utf-8'))
rpt_aug = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
rpt_winner = 'o3'
RPT_JUDGE_PROMPT = """당신은 공공 SW '사업추진결과보고서(RPT)' 검토 학습데이터 품질 검수기다.
입력은 증강 케이스 여러 건의 JSON 배열이다.
각 건을 독립적으로 재판정하고, output.eval과 output.label이 발췌문 근거와 일치하는지 검사한다.
출력은 JSON 배열만 허용한다.

[중요 원칙]
- 판단은 반드시 각 건의 pep_excerpt와 rpt_excerpt에만 근거한다.
- 외부 지식으로 없는 과업, 수치, 산출물, 기준을 보완하지 않는다.
- 법조문, refs, disclaimer 유무는 검사 기준이 아니다.
- input.criteria에 포함된 특성만 재판정한다.
- 먼저 스스로 재판정한 뒤, 그 결과를 output.eval 및 output.label과 대조한다.
- 정확성은 검토를 허용하지 않는다.

[입력 eval 파싱 규칙]
- output.eval은 문자열 배열이다.
- 각 원소는 반드시 "특성: 판정 — 사유" 형식이어야 한다.
- 각 문자열에서 특성명, 판정값, 사유를 파싱하여 검사한다.
- 특성명은 input.criteria에 포함된 값만 허용한다.
- 정확성에 "검토"가 있으면 형식 오류이자 판정 오류다.

[특성별 재판정 규칙]

1. 완전성(COMP)
- PEP의 과업, 범위, 산출물 항목마다 RPT에 대응이 있는지 확인한다.
- 동일 표현뿐 아니라 명백한 동의어와 패러프레이즈도 인정한다.
- 같은 산출물, 같은 주체, 같은 시점을 가리키는 것이 명백하면 충족이다.
- 하나라도 유사성조차 없거나 조건부 문구로만 처리되면 불가이다.
- 표현은 다르지만 같은 산출물, 같은 주체, 같은 시점인지 판단이 갈리면 검토이다.

2. 정확성(ACC)
- PEP의 계획값, 기준값, 기간, 예산, 건수, 규격과 RPT의 결과값이 직접 대조되어 일치하면 충족이다.
- 다음 중 하나라도 해당하면 불가이다.
  1) PEP 기준값과 RPT 결과값이 다름
  2) RPT 내부 자기모순
  3) 비교 가능한 기준값 부재
- 정확성은 검토를 사용하지 않는다.
- output.eval에 정확성: 검토가 있으면 그 자체로 형식 결함이자 판정 불일치로 본다.

3. 검증가능성(VER)
- 블랙리스트 표현 예:
  "적절히", "최선을 다해", "원활히 협의하여", "필요한 범위에서", "우수한 수준으로", "성실히", "무리 없이", "신속히"
- 검증 가능한 기준(수치, 건수, 기준일, 비율, 시험결과, 검수결과, 측정값 등)이 전혀 없거나 항목이 통째로 비어 있으면 불가이다.
- 블랙리스트 표현이 전혀 없고 화이트리스트 형태로 확인 가능하게 서술되면 충족이다.
- 블랙리스트 표현이 하나라도 있으면 무조건 검토이다.
  블랙리스트 표현을 제거했을 때 남는 내용으로 유추 가능한지 여부는 판정에 영향을 주지 않는다.
  즉 "독립 검증 가능하니 충족" 판정으로 격상시키지 않는다.

4. 추적성(TRACE)
- PEP의 단계, 과업, 산출물, 일정, 목표와 RPT 결과의 연결을 발췌문 안에서 확인한다.
- 명확한 1:1 대응이 확인되면 충족이다.
- 대응 항목이 없거나 연결이 끊기면 불가이다.
- 단계명, 산출물명, 구성 방식이 달라 같은 항목인지 확정하기 어렵거나 현재 발췌문만으로 연결 정보가 부족하면 검토이다.
- 아래 3신호는 보조 판단 기준으로 활용할 수 있다.
  ① 산출물명 겹침
  ② 순번·개수 대응
  ③ 기능 설명 일치
- 3신호는 참고용이다. 반드시 3개가 모두 있어야 하는 것은 아니다.

[label 재계산 규칙]
- 특성별 재판정 중 불가가 1개 이상이면 재계산label = "불가"
- 불가가 없고 검토가 1개 이상이면 재계산label = "검토"
- 모든 특성이 충족이면 재계산label = "충족"

[검사 항목]

1. 판정일치
- 스스로 재판정한 결과가 output.eval에서 파싱한 각 특성 판정과 같은가

2. label일치
- 재계산label이 output.label과 같은가

3. 사유진위
- output.eval의 사유가 실제 발췌문에 근거하는가
- 없는 대응, 없는 누락, 없는 수치불일치, 없는 자기모순, 없는 단절을 지어내지 않았는가

4. 방향성정상
- 판정과 사유가 서로 모순되지 않는가
- 예: 판정은 불가인데 사유는 "모든 항목이 대응된다"라고 쓰면 false

5. criteria보존
- input.criteria에 있는 특성만 output.eval에 존재하는가
- 빠진 특성이나 추가된 특성이 없는가
- 정확성에 검토가 들어가 있지 않은가

6. 형식정상
- 필수 필드가 모두 존재하는가
  - id
  - input.criteria
  - input.pep_excerpt
  - input.rpt_excerpt
  - output.label
  - output.eval
- output.eval의 각 원소가 "특성: 판정 — 사유" 형식인가
- 각 판정값이 허용된 값만 쓰였는가

[출력 형식 — 반드시 아래 JSON 객체 하나로, results 배열에 입력 건수만큼 모두 담아라]
{
  "results": [
    {
      "id": "...",
      "통과": true,
      "특성별재판정": { "완전성": "충족|불가|검토", "정확성": "충족|불가", "검증가능성": "충족|불가|검토", "추적성": "충족|불가|검토" },
      "재계산label": "충족|불가|검토",
      "판정일치": true,
      "label일치": true,
      "사유진위": true,
      "방향성정상": true,
      "criteria보존": true,
      "형식정상": true,
      "문제점": [],
      "신뢰도": 0.0
    }
  ]
}
반드시 입력의 모든 건을 results 배열에 포함하라. 한 건만 반환하지 마라.

[출력 제약]
- 특성별재판정에는 해당 건의 input.criteria에 있는 특성만 넣는다.
- 문제점에는 실제 불일치나 형식 결함 사유를 짧게 적는다.
- 신뢰도는 0.0~1.0 사이 값으로 주며, 발췌문 근거가 직접적이고 명확할수록 높인다.

[통과 조건]
아래가 모두 true일 때만 통과다.
- 판정일치
- label일치
- 사유진위
- 방향성정상
- criteria보존
- 형식정상

[입력]
{증강 레코드 JSON 배열}
"""
run_qc(rpt_aug, RPT_JUDGE_PROMPT, rpt_winner, 'RPT')